# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [282]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [283]:
# data loading 
df=pd.read_csv("AviationData.csv",index_col=0, encoding="latin-1")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_21536\3338872839.py:2: DtypeWarning: Columns (0: Longitude, 1: Airport.Code, 2: Report.Status) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("AviationData.csv",index_col=0, encoding="latin-1")


In [284]:
#check the shape of the data 
df.shape

(88889, 30)

In [285]:
#displays column names, data types, and non-null counts
df.info()

<class 'pandas.DataFrame'>
Index: 88889 entries, 20001218X45444 to 20221230106513
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Investigation.Type      88889 non-null  str    
 1   Accident.Number         88889 non-null  str    
 2   Event.Date              88889 non-null  str    
 3   Location                88837 non-null  str    
 4   Country                 88663 non-null  str    
 5   Latitude                34382 non-null  object 
 6   Longitude               34373 non-null  object 
 7   Airport.Code            50132 non-null  str    
 8   Airport.Name            52704 non-null  str    
 9   Injury.Severity         87889 non-null  str    
 10  Aircraft.damage         85695 non-null  str    
 11  Aircraft.Category       32287 non-null  str    
 12  Registration.Number     87507 non-null  str    
 13  Make                    88826 non-null  str    
 14  Model                   88797 no

In [286]:
#counting missing values in each column
df.isna().sum()

Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13771
dtype: int64

In [287]:
# get the summary statistics
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [288]:
df.columns

Index(['Investigation.Type', 'Accident.Number', 'Event.Date', 'Location',
       'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name',
       'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category',
       'Registration.Number', 'Make', 'Model', 'Amateur.Built',
       'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Schedule',
       'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date'],
      dtype='str')

In [289]:
#getting the percentage of the missing values and dorting them in descending order
(round(df.isna().sum()/len(df),4)*100).sort_values(ascending=False)

Schedule                  85.85
Air.carrier               81.27
FAR.Description           63.97
Aircraft.Category         63.68
Longitude                 61.33
Latitude                  61.32
Airport.Code              43.60
Airport.Name              40.71
Broad.phase.of.flight     30.56
Publication.Date          15.49
Total.Serious.Injuries    14.07
Total.Minor.Injuries      13.42
Total.Fatal.Injuries      12.83
Engine.Type                7.98
Report.Status              7.18
Purpose.of.flight          6.97
Number.of.Engines          6.84
Total.Uninjured            6.65
Weather.Condition          5.05
Aircraft.damage            3.59
Registration.Number        1.55
Injury.Severity            1.12
Country                    0.25
Amateur.Built              0.11
Model                      0.10
Make                       0.07
Location                   0.06
Accident.Number            0.00
Event.Date                 0.00
Investigation.Type         0.00
dtype: float64

In [290]:
#describe numerical columns
df.describe(include="number")

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


In [291]:
#describe categorical columns
df.describe(include="O").T

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_21536\2177166514.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="O").T


,count,unique,top,freq
Investigation.Type,88889,2,Accident,85015
Accident.Number,88889,88863,ERA22LA103,2
Event.Date,88889,14782,1982-05-16,25
Location,88837,27758,"ANCHORAGE, AK",434
Country,88663,219,United States,82248
Latitude,34382,25592,332739N,19
Longitude,34373,27156,0112457W,24
Airport.Code,50132,10374,NONE,1488
Airport.Name,52704,24870,Private,240
Injury.Severity,87889,109,Non-Fatal,67357


In [292]:
(round(df.isna().sum()/len(df),4)*100).sort_values(ascending=False)

Schedule                  85.85
Air.carrier               81.27
FAR.Description           63.97
Aircraft.Category         63.68
Longitude                 61.33
Latitude                  61.32
Airport.Code              43.60
Airport.Name              40.71
Broad.phase.of.flight     30.56
Publication.Date          15.49
Total.Serious.Injuries    14.07
Total.Minor.Injuries      13.42
Total.Fatal.Injuries      12.83
Engine.Type                7.98
Report.Status              7.18
Purpose.of.flight          6.97
Number.of.Engines          6.84
Total.Uninjured            6.65
Weather.Condition          5.05
Aircraft.damage            3.59
Registration.Number        1.55
Injury.Severity            1.12
Country                    0.25
Amateur.Built              0.11
Model                      0.10
Make                       0.07
Location                   0.06
Accident.Number            0.00
Event.Date                 0.00
Investigation.Type         0.00
dtype: float64

In [293]:
#get a copy of the dataframe 
data=df.copy()

#convert the date into proper datetime format
from datetime import datetime
data["Publication.Date"]=pd.to_datetime(data["Publication.Date"],errors="coerce")
data["Event.Date"]=pd.to_datetime(data['Event.Date'],errors="coerce")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_21536\1659456149.py:6: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  data["Publication.Date"]=pd.to_datetime(data["Publication.Date"],errors="coerce")


In [294]:
# this cell contained filtered data specifically needed by the client,

filtered_data=data[
                   (data["Event.Date"]>=pd.to_datetime("1983-01-01")) &
                   (data["Amateur.Built"]=="No")&
                   (data["Aircraft.Category"]=="Airplane")
                   ]

In [295]:
filtered_data.head()

,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,Aircraft.Category,Registration.Number,Make,Model,Amateur.Built,Number.of.Engines,Engine.Type,FAR.Description,Schedule,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
Event.Id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
20001214X42478,Incident,LAX83IA149B,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,Incident,Minor,Airplane,N323EA,Lockheed,L-1011,No,3.0,Turbo Fan,Part 121: Air Carrier,SCHD,Unknown,NaN,NaN,NaN,NaN,588.0,VMC,Standing,Probable Cause,2014-12-04
20001214X42478,Incident,LAX83IA149A,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,Incident,Minor,Airplane,9VSQQ,Boeing,747,No,4.0,Turbo Fan,Part 129: Foreign,SCHD,NaN,"Singapore Airlines, Ltd.",NaN,NaN,NaN,588.0,VMC,Taxi,Probable Cause,2014-12-04
20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,NaN,NaN,NaN,NaN,Fatal(1),Destroyed,Airplane,N9600W,Piper,PA-28-140,No,1.0,Reciprocating,Part 91: General Aviation,NaN,Personal,NaN,1.0,1.0,NaN,NaN,IMC,Cruise,Probable Cause,2011-05-02
20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,NaN,NaN,TX05,AERO COUNTRY,Fatal(1),NaN,Airplane,N927BA,De Havilland,DHC-6,No,2.0,Turbo Prop,Part 91: General Aviation,NaN,Skydiving,NaN,1.0,NaN,NaN,4.0,VMC,Standing,Probable Cause,2016-10-17
20001214X44248,Incident,MIA83IA210,1983-08-21,"NORFOLK, VA",United States,NaN,NaN,NaN,NaN,Incident,Minor,Airplane,N69NA,Douglas,DC-10-10,No,3.0,Turbo Fan,Part 121: Air Carrier,SCHD,Unknown,NaN,NaN,NaN,NaN,289.0,VMC,Cruise,Probable Cause,2016-02-01


1. The data above is skewed,,decided to use median to remove the null values.

In [296]:
#get the median for injuries then use it to replace the null values 
med_fatal=filtered_data['Total.Fatal.Injuries'].median()
med_Uninjured=filtered_data['Total.Uninjured'].median()
med_sereous=filtered_data["Total.Serious.Injuries"].median()
med_minor=filtered_data["Total.Minor.Injuries"].median()


In [297]:
#lets fill the null values using median above 
filtered_data['Total.Fatal.Injuries']=filtered_data['Total.Fatal.Injuries'].fillna(med_fatal)
filtered_data['Total.Uninjured']  =filtered_data['Total.Uninjured'].fillna(med_Uninjured)
filtered_data["Total.Serious.Injuries"]=filtered_data["Total.Serious.Injuries"].fillna(med_sereous)
filtered_data["Total.Minor.Injuries"]=filtered_data["Total.Minor.Injuries"].fillna(med_minor)



### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [298]:
#creating a column to find  total number of passangers 
filtered_data["Total_Passangers"]=(filtered_data['Total.Fatal.Injuries']+
                                   filtered_data["Total.Minor.Injuries"]+
                                   filtered_data["Total.Serious.Injuries"]+
                                   filtered_data['Total.Uninjured']
                                   )


In [299]:
#Metric for fatal/serious injured passangers 
filtered_data["Fatal_Serious_Inj"]=(filtered_data['Total.Fatal.Injuries'])+ filtered_data["Total.Serious.Injuries"]/filtered_data["Total_Passangers"]


# To avoid devision by 0
filtered_data["Fatal_Serious_Inj"]=filtered_data["Fatal_Serious_Inj"].fillna(0)

In [300]:
filtered_data = filtered_data[filtered_data["Total_Passangers"] > 0]

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [301]:
#checking the percentage null values
round(filtered_data["Aircraft.damage"].isna()/len(data),4)* 100

Event.Id
20001214X42478    0.0
20001214X42478    0.0
20001214X42331    0.0
20001214X42672    0.0
20001214X44248    0.0
                 ... 
20221213106455    0.0
20221215106463    0.0
20221219106475    0.0
20221219106470    0.0
20221227106497    0.0
Name: Aircraft.damage, Length: 20590, dtype: float64

In [302]:
# remove white spaces 
filtered_data["Aircraft.damage"]=(filtered_data["Aircraft.damage"]).astype(str).str.strip()


In [303]:
#To check the aircraft destroyed
filtered_data["Aircraft_Destroyed"] = filtered_data["Aircraft.damage"].apply(
    lambda x: 1 if x == "Destroyed" else 0
)
filtered_data[["Aircraft.damage", "Aircraft_Destroyed"]].head()

,Aircraft.damage,Aircraft_Destroyed
Event.Id,,
20001214X42478,Minor,0
20001214X42478,Minor,0
20001214X42331,Destroyed,1
20001214X42672,NaN,0
20001214X44248,Minor,0


In [304]:
#After checking there are 1227 null values in the column Aircraft.damage
filtered_data["Aircraft.damage"].isna().sum()

#use fillna to replace the missing values with unknown.
filtered_data["Aircraft.damage"]=filtered_data["Aircraft.damage"].fillna("unknown", inplace=True)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_21536\4172604795.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  filtered_data["Aircraft.damage"]=filtered_data["Aircraft.damage"].fillna("unknown", inplace=True)


In [305]:
#To confirm if there is still null values.
#There is none
filtered_data["Aircraft.damage"].isna().sum()

np.int64(0)

In [306]:
#Yes, we dont have any null values in the column
filtered_data["Aircraft.damage"].unique()

<StringArray>
['Minor', 'Destroyed', 'unknown', 'Substantial', 'Unknown']
Length: 5, dtype: str

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [307]:
# Below we convert  the values in column make to lower case for uniformity
filtered_data['Make']=(filtered_data["Make"]
                       .astype(str) #convert the column into a string type
                       .str.strip() # Remove white spaces 
                       .str.lower()) #change to lowercase

In [308]:
#Check the null values in the column ,Make
filtered_data["Make"].isna().sum()

#There is only one null value, we can fill it with unknown 
filtered_data["Make"]=filtered_data["Make"].fillna("unknown", inplace=True)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_21536\1926944182.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  filtered_data["Make"]=filtered_data["Make"].fillna("unknown", inplace=True)


In [309]:
filtered_data["Make"].isna().sum()

np.int64(0)

In [310]:
make_counts = filtered_data["Make"].value_counts()

# Identify the makes to keep 
valid_makes=make_counts[make_counts >=50].index

pd.set_option("display.max_columns", None)

#This keeps the rows where Make value is in valid makes
makes_filtered=filtered_data[filtered_data["Make"].isin(valid_makes)]
makes_filtered.head()


,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,Aircraft.Category,Registration.Number,Make,Model,Amateur.Built,Number.of.Engines,Engine.Type,FAR.Description,Schedule,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total_Passangers,Fatal_Serious_Inj,Aircraft_Destroyed
Event.Id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
20001214X42478,Incident,LAX83IA149A,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,Incident,Minor,Airplane,9VSQQ,boeing,747,No,4.0,Turbo Fan,Part 129: Foreign,SCHD,NaN,"Singapore Airlines, Ltd.",0.0,0.0,0.0,588.0,VMC,Taxi,Probable Cause,2014-12-04,588.0,0.000000,0
20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,NaN,NaN,NaN,NaN,Fatal(1),Destroyed,Airplane,N9600W,piper,PA-28-140,No,1.0,Reciprocating,Part 91: General Aviation,NaN,Personal,NaN,1.0,1.0,0.0,1.0,IMC,Cruise,Probable Cause,2011-05-02,3.0,1.333333,1
20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,NaN,NaN,TX05,AERO COUNTRY,Fatal(1),unknown,Airplane,N927BA,de havilland,DHC-6,No,2.0,Turbo Prop,Part 91: General Aviation,NaN,Skydiving,NaN,1.0,0.0,0.0,4.0,VMC,Standing,Probable Cause,2016-10-17,5.0,1.000000,0
20001214X45013,Incident,CHI84IA041,1983-11-08,"CHICAGO, IL",United States,NaN,NaN,ORD,O'HARE,Incident,Minor,Airplane,N898AA,boeing,727-200,No,3.0,Turbo Fan,Part 121: Air Carrier,SCHD,Unknown,NaN,0.0,0.0,0.0,100.0,VMC,Taxi,Probable Cause,2018-06-11,100.0,0.000000,0
20001214X45188,Accident,NYC84LA028,1983-11-13,"MARTHA'S VINEYARD, MA",United States,NaN,NaN,NaN,NaN,Non-Fatal,Substantial,Airplane,N1882D,beech,C35,No,1.0,Reciprocating,Part 91: General Aviation,NaN,Personal,NaN,0.0,0.0,0.0,1.0,VMC,Climb,Probable Cause,2011-05-05,1.0,0.000000,0


In [311]:
#Checking the shape of the new dataframe,,(it has 17892 rows/records and 33 columns)
makes_filtered.shape

(17113, 33)

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [312]:
#Checking the null values in column model
makes_filtered["Model"].isna().sum()

#I decided to fill the null values with unknown
makes_filtered["Model"]=makes_filtered["Model"].fillna("unkown", inplace=True)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_21536\3794119253.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  makes_filtered["Model"]=makes_filtered["Model"].fillna("unkown", inplace=True)


In [313]:
#check how many times each make appears 
makes_filtered[["Make"]].value_counts().head(20)

Make              
cessna                7067
piper                 3960
beech                 1416
boeing                 839
mooney                 360
cirrus design corp     220
bellanca               219
air tractor inc        219
maule                  215
air tractor            203
aeronca                200
champion               158
grumman                147
luscombe               141
airbus                 133
cirrus                 133
embraer                130
stinson                129
north american         105
dehavilland             95
Name: count, dtype: int64

In [314]:
#cehck how many times does the model appear
filtered_data["Model"].value_counts().head(20)

Model
172          760
152          313
182          301
172S         274
PA28         267
SR22         263
172N         247
180          213
737          199
A36          189
PA-18-150    181
172M         180
150          176
PA-28-140    169
172P         142
140          117
172R         108
170B         107
PA-28-161    106
PA-28-180    105
Name: count, dtype: int64

#### have decided to group make and model to see thier relationship

In [315]:
makes_filtered.groupby(["Make", "Model"]).size().sort_values(ascending=False).head(20)


Make                Model    
cessna              172          757
                    152          312
                    182          301
                    172S         272
piper               PA28         267
cessna              172N         246
                    180          213
boeing              737          199
cessna              172M         180
                    150          176
piper               PA-18-150    175
                    PA-28-140    169
beech               A36          164
cirrus design corp  SR22         144
cessna              172P         142
                    140          116
                    172R         107
                    170B         107
piper               PA-28-180    105
                    PA-28-161    102
dtype: int64

#### Observation: Models are not always unique to single make

In [316]:
#Now there is no null values 
makes_filtered["Model"].isna().sum()

np.int64(0)

In [317]:
makes_filtered["Make_Model"] = (
    makes_filtered["Make"].astype(str).str.strip().str.lower()
    + "_" +
   makes_filtered["Model"].astype(str).str.strip().str.lower()
)

In [318]:
makes_filtered[["Make", "Model","Make_Model"]]

,Make,Model,Make_Model
Event.Id,,,
20001214X42478,boeing,747,boeing_747
20001214X42331,piper,PA-28-140,piper_pa-28-140
20001214X42672,de havilland,DHC-6,de havilland_dhc-6
20001214X45013,boeing,727-200,boeing_727-200
20001214X45188,beech,C35,beech_c35
...,...,...,...
20221212106444,cessna,172,cessna_172
20221213106455,piper,PA42,piper_pa42
20221215106463,cirrus design corp,SR22,cirrus design corp_sr22


In [319]:
# Check if a single model appears under multiple makes
model_make_check = (
    makes_filtered.groupby("Model")["Make"]
    .nunique()
    .sort_values(ascending=False)
)

model_make_check.head(20)

Model
unkown      5
7EC         3
8GCBC       3
8KCAB       3
7AC         3
500         3
7ECA        3
7GCAA       3
7GCBC       3
S2R         3
402A        2
402B        2
A-1         2
7KCAB       2
A-1A        2
A-1B        2
B200        2
AT-502B     2
A-1C-180    2
A-1C-200    2
Name: Make, dtype: int64

In [320]:
makes_filtered.notna().sum()

Investigation.Type        17113
Accident.Number           17113
Event.Date                17113
Location                  17109
Country                   17112
Latitude                  15790
Longitude                 15787
Airport.Code              11443
Airport.Name              11544
Injury.Severity           17113
Aircraft.damage           17113
Aircraft.Category         17113
Registration.Number       16953
Make                      17113
Model                     17113
Amateur.Built             17113
Number.of.Engines         15537
Engine.Type               14489
FAR.Description           16892
Schedule                   1784
Purpose.of.flight         14802
Air.carrier                8018
Total.Fatal.Injuries      17113
Total.Serious.Injuries    17113
Total.Minor.Injuries      17113
Total.Uninjured           17113
Weather.Condition         15371
Broad.phase.of.flight      2453
Report.Status             13992
Publication.Date          16508
Total_Passangers          17113
Fatal_Se

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [321]:
#get the percentage null values 
round(makes_filtered[['Weather.Condition',
                'Number.of.Engines', 
                'Purpose.of.flight',
                'Broad.phase.of.flight',
                'Engine.Type'
                ]].isna().sum()/len(data),4)*100

Weather.Condition         1.96
Number.of.Engines         1.77
Purpose.of.flight         2.60
Broad.phase.of.flight    16.49
Engine.Type               2.95
dtype: float64

In [322]:
makes_filtered["Number.of.Engines"].skew()


np.float64(2.7185903612393707)

## Number of engines 

In [323]:
#The data is right skewed. for this case am going to use median to get fill thye null values.
makes_filtered["Number.of.Engines"].skew()

#get the median 
num_of_eng=makes_filtered['Number.of.Engines'].median()

#fill tthe null values using the above median
makes_filtered["Number.of.Engines"]=makes_filtered['Number.of.Engines'].fillna(num_of_eng, inplace=True)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_21536\2864425188.py:8: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  makes_filtered["Number.of.Engines"]=makes_filtered['Number.of.Engines'].fillna(num_of_eng, inplace=True)


## Weather condition

In [324]:
#For this case i changed it to upper to keep the consistency in the abbreviations. 
makes_filtered["Weather.Condition"] = (
    makes_filtered["Weather.Condition"]
    .astype(str)
    .str.strip()
    .str.upper()
    .replace(["UNK", "UNKN", "NAN", ""], pd.NA)
)

In [325]:
#  For this case i just filled the null values using unknown.
makes_filtered["Weather.Condition"]=makes_filtered["Weather.Condition"].fillna("unknown")

In [326]:
makes_filtered["Weather.Condition"].unique()

<StringArray>
['VMC', 'IMC', 'unknown']
Length: 3, dtype: str

## Purpose of flight

In [327]:
# Removed white spaces and inconsistency capitaliation
makes_filtered["Purpose.of.flight"] = (
    makes_filtered["Purpose.of.flight"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(["nan", "none", ""], pd.NA)
)

In [328]:
# fill the null values in the purpose of flight using unknown
makes_filtered["Purpose.of.flight"]=makes_filtered["Purpose.of.flight"].fillna("unknown")

In [329]:
makes_filtered["Purpose.of.flight"].unique()

<StringArray>
[                  'unknown',                  'personal',
                 'skydiving',        'aerial application',
               'positioning',             'instructional',
                  'business',           'public aircraft',
                     'ferry',            'other work use',
        'aerial observation',       'executive/corporate',
 'public aircraft - federal',             'air race/show',
               'flight test',   'public aircraft - state',
                'glider tow',                'banner tow',
             'air race show',              'firefighting',
   'public aircraft - local',                  'air drop',
                      'pubs',                      'asho']
Length: 24, dtype: str

## Engine type

In [330]:
makes_filtered["Engine.Type"].unique()

<StringArray>
[      'Turbo Fan',   'Reciprocating',      'Turbo Prop',       'Turbo Jet',
               nan,         'Unknown',     'Turbo Shaft', 'Geared Turbofan',
             'UNK']
Length: 9, dtype: str

In [331]:
# Engine type cleaning, removed white spaces, converted it to lower case
makes_filtered["Engine.Type"] = (
    makes_filtered["Engine.Type"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(["nan", "none", ""], pd.NA)
)

In [332]:
#filling the null values 
makes_filtered["Engine.Type"]=makes_filtered["Engine.Type"].fillna("unknown")

## Broad phase of flight    

In [333]:
# worked on:
#inconsistent formatting,
#missing values and 
#duplicates due to casing

makes_filtered["Broad.phase.of.flight"] = (
    makes_filtered["Broad.phase.of.flight"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(["nan", "none", ""], pd.NA)
)

In [334]:
makes_filtered["Broad.phase.of.flight"]=makes_filtered["Broad.phase.of.flight"].fillna("unknown")

In [335]:
makes_filtered["Broad.phase.of.flight"].unique()

<StringArray>
[       'taxi',      'cruise',    'standing',       'climb',     'takeoff',
     'landing',    'approach', 'maneuvering',     'descent',     'unknown',
   'go-around',       'other']
Length: 12, dtype: str

In [336]:
#I used the round function to check if we still have any peercentage null values
#but we had none.
round(makes_filtered[['Weather.Condition',
                'Number.of.Engines', 
                'Purpose.of.flight',
                'Broad.phase.of.flight',
                'Engine.Type'
                ]].isna().sum()/len(data),4)*100

Weather.Condition        0.0
Number.of.Engines        0.0
Purpose.of.flight        0.0
Broad.phase.of.flight    0.0
Engine.Type              0.0
dtype: float64

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [337]:
#Drop the unnecessary columns
makes_filtered=makes_filtered.drop(['Longitude',
                                    'Latitude',
                                    'Airport.Code',
                                    'Airport.Name',
                                    'Schedule',
                                     'Country',
                                     'Event.Date',
                                     'Publication.Date',
                                     'Registration.Number',
                                     'Air.carrier',
                                     'FAR.Description',
                                      'Purpose.of.flight',
                                    
                                    ], axis=1)



In [338]:
# For this case i noticed there are some null values in this column,,, i had to fill them with unknown.
makes_filtered["Report.Status"]=makes_filtered["Report.Status"].fillna("unknown")

In [339]:
round(makes_filtered.isna().sum()/len(data),4)*100

Investigation.Type        0.0
Accident.Number           0.0
Location                  0.0
Injury.Severity           0.0
Aircraft.damage           0.0
Aircraft.Category         0.0
Make                      0.0
Model                     0.0
Amateur.Built             0.0
Number.of.Engines         0.0
Engine.Type               0.0
Total.Fatal.Injuries      0.0
Total.Serious.Injuries    0.0
Total.Minor.Injuries      0.0
Total.Uninjured           0.0
Weather.Condition         0.0
Broad.phase.of.flight     0.0
Report.Status             0.0
Total_Passangers          0.0
Fatal_Serious_Inj         0.0
Aircraft_Destroyed        0.0
Make_Model                0.0
dtype: float64

In [340]:
#I checked the nun null values, there wasnt any non-null value above 20,000.
makes_filtered.notna().sum().sort_values(ascending=False)

Investigation.Type        17113
Accident.Number           17113
Injury.Severity           17113
Aircraft.damage           17113
Aircraft.Category         17113
Make                      17113
Total.Uninjured           17113
Model                     17113
Amateur.Built             17113
Number.of.Engines         17113
Engine.Type               17113
Total.Fatal.Injuries      17113
Total.Serious.Injuries    17113
Total.Minor.Injuries      17113
Total_Passangers          17113
Weather.Condition         17113
Broad.phase.of.flight     17113
Report.Status             17113
Aircraft_Destroyed        17113
Fatal_Serious_Inj         17113
Make_Model                17113
Location                  17109
dtype: int64

In [341]:
makes_filtered.shape

(17113, 22)

In [342]:
#Here,, i changed all the column names to lower case, and replaced "." with "_"
makes_filtered.columns = makes_filtered.columns.str.lower().str.replace('.', '_', regex=False)

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [343]:
makes_filtered.to_csv("cleaned_airline_df.csv", index=None)